In [6]:
# 1. Imports
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_curve, roc_auc_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder
from onnxmltools import convert_lightgbm
from onnxmltools.convert.common.data_types import FloatTensorType
from matplotlib import pyplot as plt
from pathlib import Path

# 2. Load data
DATA_DIR = Path("../data")
tx = pd.read_csv(DATA_DIR / "train_transaction.csv")
id_df = pd.read_csv(DATA_DIR / "train_identity.csv")
df = tx.merge(id_df, on="TransactionID", how="left")
print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {df['isFraud'].mean():.4%}")

# 3. Feature selection — all features
drop_cols = ["isFraud", "TransactionID"]
FEATURES = [col for col in df.columns if col not in drop_cols]

df = df[FEATURES + ["isFraud"]].copy()

# 4. Encode ALL string columns
for col in df.columns:
    if df[col].dtype == 'object' or str(df[col].dtype) == 'str':
        le = LabelEncoder()
        df[col] = df[col].astype(str).fillna("unknown")
        df[col] = le.fit_transform(df[col])

df = df.fillna(-999)

# Verify — check both object and str dtypes
string_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
assert len(string_cols) == 0, f"String columns still present: {string_cols}"
print("All columns numeric")

# 5. Split
X = df[FEATURES]
y = df["isFraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# 6. Train
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=scale,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)])

# 7. Evaluate
y_prob = model.predict_proba(X_test)[:, 1]

OPERATING_THRESHOLD = 0.90
y_pred = (y_prob >= OPERATING_THRESHOLD).astype(int)

print(f"\nClassification report (threshold={OPERATING_THRESHOLD}):")
print(classification_report(y_test, y_pred, target_names=["legit", "fraud"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

diff = np.abs(precision[:-1] - recall[:-1])
crossover_idx = diff.argmin()
print(f"\nCrossover point:")
print(f"  Threshold: {thresholds[crossover_idx]:.4f}")
print(f"  Precision: {precision[crossover_idx]:.4f}")
print(f"  Recall:    {recall[crossover_idx]:.4f}")

print(f"\nExact metrics at threshold {OPERATING_THRESHOLD}:")
print(f"  Precision: {precision_score(y_test, y_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred):.4f}")

Dataset shape: (590540, 434)
Fraud rate: 3.4990%
All columns numeric
Train: 472432, Test: 118108
[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.143908 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 39065
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 431
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101

Classification report (threshold=0.9):
              precision    recall  f1-score   support

       legit       0.98      1.00      0.99    113975
       fraud       0.85      0.50      0.63      4133

    accuracy                           0.98    118108
   macro avg       0.92      0.75      0.81    118108
weighted avg       0.98      